# 하이퍼파라미터 실습

**Hyperparameter**

학습 과정 밖에서 정하거나 조정하는 모델·학습 설정.

소재 분야에서 이해하기: 트리 깊이를 검증 데이터로 선택한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 설정을 검증 데이터로 고릅니다

트리 깊이를 바꿔가며 교차검증 점수를 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
grid = GridSearchCV(RandomForestRegressor(random_state=0),
                    {'max_depth': [2, 4, 6, 10, None], 'n_estimators': [100, 300]},
                    cv=5, scoring='neg_mean_absolute_error').fit(X_train, y_train)
for depth, estimators, score in zip(grid.cv_results_['param_max_depth'],
                                    grid.cv_results_['param_n_estimators'],
                                    grid.cv_results_['mean_test_score']):
    print('max_depth=%-4s n_estimators=%-4s 검증 MAE %.2f' % (depth, estimators, -score))
print('\n선택된 설정:', grid.best_params_)
print('시험 데이터 MAE %.2f (설정 선택에 쓰지 않은 데이터)' % (-grid.score(X_test, y_test)))

## 2. 해석

하이퍼파라미터는 학습이 스스로 정하지 못하므로 검증 데이터로 고릅니다. 이때 시험 데이터를 쓰면
선택 과정이 시험 데이터에 맞춰져 성능이 부풀려집니다. 시험 데이터는 마지막에 한 번만 씁니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#hyperparameter)을 여세요.